In [32]:
# ==============================================================================
# KACHCHH BIRD SDM — ALL PARAMETERS, SINGLE 11.1 KM GRID (v2 — split exports)
# Optimization vs. v1: instead of one 226-band export (which failed on the
# heaviest component), each group exports SEPARATELY. All groups share the
# identical 11.1 km grid (same CRS + scale), so they stack trivially later —
# but each export is now much lighter, and a failure in one doesn't cost you
# the others.
# ==============================================================================

# CELL 1 — Install / import
!pip install -q earthengine-api

import ee
import datetime

# CELL 2 — Authenticate & initialize
ee.Authenticate()
ee.Initialize(project='guide-project-505706')

# CELL 3 — Study area: Kachchh district boundary
gaul2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2')
kachchh_fc = gaul2.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'India'),
        ee.Filter.eq('ADM1_NAME', 'Gujarat'),
        ee.Filter.Or(
            ee.Filter.eq('ADM2_NAME', 'Kachchh'),
            ee.Filter.eq('ADM2_NAME', 'Kutch'),
            ee.Filter.eq('ADM2_NAME', 'Kachch'),
        )
    )
)
print('Matched district name:', kachchh_fc.first().get('ADM2_NAME').getInfo())
kachchh_geom = kachchh_fc.geometry()

# CELL 4 — Global constants
TARGET_SCALE = 11132
TARGET_CRS = 'EPSG:4326'
CURRENT_YEAR = datetime.date.today().year
CLIMATE_START = '1973-01-01'
RECENT_END = datetime.date.today().isoformat()
RECENT_START = (datetime.date.today() - datetime.timedelta(days=3 * 365)).isoformat()

Matched district name: Kachchh


In [33]:
# CELL 5 — Aggregation helper (lighter maxPixels for reliability)
def aggregate_to_target(image, native_scale):
    return (
        image
        .reproject(crs=TARGET_CRS, scale=native_scale)
        .reduceResolution(reducer=ee.Reducer.mean(), bestEffort=True, maxPixels=4096)
        .reproject(crs=TARGET_CRS, scale=TARGET_SCALE)
    )

def export_group(image, description, prefix):
    task = ee.batch.Export.image.toDrive(
        image=image.clip(kachchh_geom).toFloat(),  # uniform type avoids band dtype-mismatch errors
        description=description,
        folder='GEE_exports',
        fileNamePrefix=prefix,
        region=kachchh_geom,
        scale=TARGET_SCALE,
        crs=TARGET_CRS,
        maxPixels=1e9,
    )
    task.start()
    print(f'{description} export started. Task ID: {task.id}')
    return task

In [34]:
# ==============================================================================
# GROUP 1 — TERRAIN, WATER, PROTECTION (native 30 m)
# ==============================================================================

# CELL 6
srtm = ee.Image('USGS/SRTMGL1_003')
elevation = srtm.select('elevation').rename('elevation')
slope_degree = ee.Terrain.slope(srtm.select('elevation')).rename('slope_degree')

gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
water_occurrence_percentage = gsw.select('occurrence').unmask(0).rename('water_occurrence_percentage')

wdpa = (
    ee.FeatureCollection('WCMC/WDPA/current/polygons')
    .filterBounds(kachchh_geom)
    .filter(ee.Filter.neq('STATUS', 'Proposed'))
)
protected_area_percentage = ee.Image(0).byte().paint(wdpa, 1).multiply(100).rename('protected_area_percentage')

terrain_water_protection = aggregate_to_target(
    ee.Image.cat([elevation, slope_degree, water_occurrence_percentage, protected_area_percentage]),
    native_scale=30,
).toFloat()



In [35]:
# CELL 7 — Export Group 1
task_terrain = export_group(terrain_water_protection, 'Kachchh_TerrainWaterProtection', 'kachchh_terrain_water_protection_11km')

Kachchh_TerrainWaterProtection export started. Task ID: KE64JTP434FGY4LCWZMCSD7Y


In [58]:
# ==============================================================================
# GROUP 2 — SPECTRAL INDICES (native 10 m Sentinel-2) — the heaviest group
# ==============================================================================

# CELL 8
CS_BAND = 'cs_cdf'
CLEAR_THRESHOLD = 0.60
MAX_CLOUD_PCT = 30
S2_WORKING_SCALE = 100  # Final output is 11.1 km anyway, so native 10 m precision
                        # isn't needed — downsampling BEFORE compositing (median)
                        # cuts pixel count ~100x, which is what was causing OOM.

s2_sr = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(RECENT_START, RECENT_END)
    .filterBounds(kachchh_geom)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
)
print('Sentinel-2 scenes after cloud pre-filter:', s2_sr.size().getInfo())

cloud_score_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
s2_masked = (
    s2_sr.linkCollection(cloud_score_plus, [CS_BAND])
    .map(lambda img: img.updateMask(img.select(CS_BAND).gte(CLEAR_THRESHOLD)))
    .map(lambda img: img.reproject(crs=TARGET_CRS, scale=S2_WORKING_SCALE))
)
s2_median = s2_masked.select(['B3', 'B4', 'B8', 'B11']).median()

ndvi = aggregate_to_target(s2_median.normalizedDifference(['B8', 'B4']).rename('NDVI'), native_scale=S2_WORKING_SCALE)
ndwi = aggregate_to_target(s2_median.normalizedDifference(['B3', 'B8']).rename('NDWI'), native_scale=S2_WORKING_SCALE)
mndwi = aggregate_to_target(s2_median.normalizedDifference(['B3', 'B11']).rename('MNDWI'), native_scale=S2_WORKING_SCALE)



Sentinel-2 scenes after cloud pre-filter: 1971


In [74]:
# CELL 9 — Export Group 2, one task per index (lighter, isolates failures)
task_ndvi = export_group(ndvi, 'Kachchh_NDVI', 'kachchh_ndvi_11km')
#task_ndwi = export_group(ndwi, 'Kachchh_NDWI', 'kachchh_ndwi_11km')
#task_mndwi = export_group(mndwi, 'Kachchh_MNDWI', 'kachchh_mndwi_11km')

Kachchh_NDVI export started. Task ID: KS7BUPCEXD6TDQ3YYUC42CUP


In [38]:
# ==============================================================================
# GROUP 3 — HUMAN FOOTPRINT (native 100 m)
# ==============================================================================

# CELL 10
ghsl_img = (
    ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
    .filterDate('1975-01-01', '2020-12-31')
    .filterBounds(kachchh_geom)
    .sort('system:time_start', False)
    .first()
)
built_up_percentage = ghsl_img.select('built_surface').divide(10000).multiply(100).rename('built_up_percentage')

worldpop_img = (
    ee.ImageCollection('WorldPop/GP/100m/pop')
    .filterBounds(kachchh_geom)
    .sort('system:time_start', False)
    .first()
)
population_density = worldpop_img.select('population').multiply(100).rename('population_density_people_per_km2')

human_footprint = aggregate_to_target(
    ee.Image.cat([built_up_percentage, population_density]), native_scale=100
)


In [39]:
# CELL 11 — Export Group 3
task_human = export_group(human_footprint, 'Kachchh_HumanFootprint', 'kachchh_human_footprint_11km')

Kachchh_HumanFootprint export started. Task ID: FNSNPAUFQHQ5YGFHORYUTW4Q


In [40]:
# ==============================================================================
# GROUP 4 — NIGHT LIGHTS (native ~464 m)
# ==============================================================================

# CELL 12
viirs_recent = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
    .filterDate(RECENT_START, RECENT_END)
    .filterBounds(kachchh_geom)
    .select('avg_rad')
)
night_lights_radiance = aggregate_to_target(
    viirs_recent.median().rename('night_lights_radiance'), native_scale=463.83
)

In [41]:
# CELL 13 — Export Group 4
task_night = export_group(night_lights_radiance, 'Kachchh_NightLights', 'kachchh_night_lights_11km')

Kachchh_NightLights export started. Task ID: N4G6V2CADTRA66LRLPFYD6DX


In [42]:
# ==============================================================================
# GROUP 5 — CLIMATE, YEARLY MEANS 1973-CURRENT (native 11.1 km — lightest group)
# ==============================================================================

# CELL 14
era5 = (
    ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
    .filterDate(CLIMATE_START, RECENT_END)
    .filterBounds(kachchh_geom)
)

def to_derived_bands(img):
    date = ee.Date(img.get('system:time_start'))
    days_in_month = date.advance(1, 'month').difference(date, 'day')
    rainfall = img.select('total_precipitation_sum').multiply(1000).divide(days_in_month).rename('rainfall_mm_day')
    temperature = img.select('temperature_2m').subtract(273.15).rename('temperature_C')
    solar = img.select('surface_solar_radiation_downwards_sum').divide(1e6).divide(days_in_month).rename('solar_radiation_MJ_m2_day')
    wind = img.expression(
        'sqrt(u * u + v * v)',
        {'u': img.select('u_component_of_wind_10m'), 'v': img.select('v_component_of_wind_10m')},
    ).rename('wind_speed_m_s')
    return ee.Image.cat([rainfall, temperature, solar, wind]).copyProperties(img, ['system:time_start'])

climate_monthly = era5.map(to_derived_bands)
BASE_CLIMATE_BANDS = ['rainfall_mm_day', 'temperature_C', 'solar_radiation_MJ_m2_day', 'wind_speed_m_s']

def yearly_mean_image(year):
    year_collection = climate_monthly.filterDate(f'{year}-01-01', f'{year + 1}-01-01')
    renamed = [f'{b}_{year}' for b in BASE_CLIMATE_BANDS]
    return year_collection.mean().rename(renamed)

years = list(range(1973, CURRENT_YEAR + 1))
climate_yearly = ee.Image.cat([yearly_mean_image(y) for y in years]).reproject(crs=TARGET_CRS, scale=TARGET_SCALE)
print(f'Climate years included: {years[0]}-{years[-1]} ({len(years)} years, last year may be partial)')


Climate years included: 1973-2026 (54 years, last year may be partial)


In [43]:
# CELL 15 — Export Group 5
task_climate = export_group(climate_yearly, 'Kachchh_ClimateYearly', 'kachchh_climate_yearly_11km')

Kachchh_ClimateYearly export started. Task ID: 63G7KCFYOUC5DNNA57GCSQD6


In [76]:
# ==============================================================================
# MONITOR ALL 5 TASKS
# ==============================================================================

# CELL 16 — Re-run this cell to check progress and see failure reasons
tasks = {
    'task_ndvi': task_ndvi,
    'task_ndwi': task_ndwi,
    'task_mndwi': task_mndwi,
}
for name, t in tasks.items():
    status = t.status()
    print(f"{name:28s} -> {status.get('state'):10s} | error: {status.get('error_message', 'None')}")

task_ndvi                    -> RUNNING    | error: None
task_ndwi                    -> FAILED     | error: Execution failed; out of memory.
task_mndwi                   -> FAILED     | error: Execution failed; out of memory.
